# Ejercicio 5: Espacio Vectorial  
**Nombre:** Alexis Bautista  
**Fecha de Entrega:** 13 de mayo del 2026

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


Carga de corpus

In [7]:
import pandas as pd

corpus = pd.read_csv('wikipedia_text_corpus.csv')

print("Corpus cargado")

Corpus cargado


Preprocesamiento

In [19]:
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# stemmer
ps = PorterStemmer()

def preprocesamiento(contenido):
    # Limpieza de caracteres especiales
    caracteres_limpiados = re.sub(r'[^\w\s]', '', contenido)
    
    # Pasar a minusculas
    caracteres_limpiados = caracteres_limpiados.lower()
    
    # Tokenizacion y stemming
    resultado = [ps.stem(word) for word in caracteres_limpiados.split()]
    
    return resultado

# aplicar preprocesamiento
corpus['text_procesado'] = corpus['text'].apply(preprocesamiento)

print("Preprocesamiento completado")

# Convertir tokens procesados de nuevo a texto para TF-IDF
corpus['text_tokens'] = corpus['text_procesado'].apply(lambda x: ' '.join(x))

Preprocesamiento completado


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# TF-IDF
vectorizador = TfidfVectorizer(stop_words='english', max_features=1000)
tfidf_matriz = vectorizador.fit_transform(corpus['text_tokens'])

# Queries
queries = [
    "machine learning artificial intelligence",
    "python programming language",
    "data science analysis",
    "natural language processing",
    "web development",
    "quantum computing",
    "climate change global warming",
    "history ancient civilizations",
    "sports football basketball",
    "music art culture"
]

print("-----------------------")
print("Recuperacion con TF-IDF")
print("-----------------------")

# Procesar y recuperar resultados para cada query
resultados_tfidf = {}

for idx, query in enumerate(queries, 1):
    # Preprocesar query 
    query_procesado = preprocesamiento(query)
    query_tokens = ' '.join(query_procesado)
    
    # Transformar query usando el vectorizador entrenado
    query_tfidf = vectorizador.transform([query_tokens])
    
    # Calcular similitud coseno
    similitudes = cosine_similarity(query_tfidf, tfidf_matriz)[0]
    
    # Obtener top 5 documentos con mayor similitud
    top_indices = np.argsort(similitudes)[-5:][::-1]
    
    resultados_tfidf[query] = {
        'indices': top_indices,
        'puntajes': similitudes[top_indices]
    }
    
    # Mostrar resultados
    print("----------------------------------")
    print(f"\nQuery {idx}: '{query}'")
    print(f"Tokens: {query_tokens}")
    print("----------------------------------")
    for rango, (doc_idx, puntaje) in enumerate(zip(top_indices, similitudes[top_indices]), 1):
        if puntaje > 0:  # Mostrara si hay similitudes
            print(f"  {rango}. Doc {doc_idx} (Puntaje: {puntaje:.4f})")
            print(f"     {corpus['text'].iloc[doc_idx][:100]}...") 

-----------------------
Recuperacion con TF-IDF
-----------------------
----------------------------------

Query 1: 'machine learning artificial intelligence'
Tokens: machin learn artifici intellig
----------------------------------
  1. Doc 10547 (Puntaje: 0.7592)
     Outline of machine learning

The following outline is provided as an overview of and topical guide t...
  2. Doc 4733 (Puntaje: 0.5932)
     Teaching machine

Teaching machines were originally mechanical devices. They presented educational m...
  3. Doc 9538 (Puntaje: 0.5210)
     DoceboLMS

Docebo is a software as a service artificial intelligence platform for e-learning, also k...
  4. Doc 3905 (Puntaje: 0.4879)
     Accounting intelligence

A specialist form of business intelligence, accounting intelligence is the ...
  5. Doc 7900 (Puntaje: 0.4833)
     Digital learning

Digital learning is any type of learning that is accompanied by technology or by i...
----------------------------------

Query 2: 'python program

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [35]:
from rank_bm25 import BM25Okapi

# corpus como lista de tokens
corpus_tokens = [doc.split() for doc in corpus['text_tokens']]

# índice BM25
bm25 = BM25Okapi(corpus_tokens)

print("-----------------------")
print("Recuperacion con BM25")
print("-----------------------")

# Procesar y recuperar resultados para cada query
resultados_bm25 = {}

for idx, query in enumerate(queries, 1):
    # Preprocesar query
    query_procesado = preprocesamiento(query)
    
    # Calcular puntajes BM25
    puntajes = bm25.get_scores(query_procesado)
    
    # Obtener los 5 documentos con mayor puntuacion
    top_indices = np.argsort(puntajes)[-5:][::-1]
    
    resultados_bm25[query] = {
        'indices': top_indices,
        'puntajes': puntajes[top_indices]
    }
    
    # Mostrar resultados
    print("----------------------------------")
    print(f"Query {idx}: '{query}'")
    print(f"Tokens: {' '.join(query_procesado)}")
    print("----------------------------------")
    for rango, (doc_idx, puntaje) in enumerate(zip(top_indices, puntajes[top_indices]), 1):
        if puntaje > 0:  # Mostrar solo si hay puntuación
            print(f"  {rango}. Doc {doc_idx} (Puntaje: {puntaje:.4f})")
            print(f"     {corpus['text'].iloc[doc_idx][:100]}...")

-----------------------
Recuperacion con BM25
-----------------------
----------------------------------
Query 1: 'machine learning artificial intelligence'
Tokens: machin learn artifici intellig
----------------------------------
  1. Doc 10547 (Puntaje: 21.6419)
     Outline of machine learning

The following outline is provided as an overview of and topical guide t...
  2. Doc 8528 (Puntaje: 19.5934)
     Trace3

Trace3, Inc. is an Irvine, CA-based Information technology (IT) company and managed service ...
  3. Doc 5991 (Puntaje: 19.0860)
     Insilico Medicine

Insilico Medicine is an American biotechnology company based in Rockville in John...
  4. Doc 9538 (Puntaje: 18.8842)
     DoceboLMS

Docebo is a software as a service artificial intelligence platform for e-learning, also k...
  5. Doc 6569 (Puntaje: 18.6432)
     Cybernetics and Systems

Cybernetics and Systems is a peer-reviewed scientific journal of cybernetic...
----------------------------------
Query 2: 'python progra

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [ ]:
print("-------------------------")
print("Comparacion de resultados")
print("-------------------------")

# Crear tabla comparativa para cada query
for idx, query in enumerate(queries, 1):
    print(f"\n-----------------------------------------------")
    print(f"QUERY {idx}: '{query}'")
    print(f"\n-----------------------------------------------")
    
    # Obtener indices y puntajes de ambos modelos
    tfidf_indices = resultados_tfidf[query]['indices']
    tfidf_puntajes = resultados_tfidf[query]['puntajes']
    
    bm25_indices = resultados_bm25[query]['indices']
    bm25_puntajes = resultados_bm25[query]['puntajes']
    
    
    print(f"\n{'TF-IDF':<50} | {'BM25':<50}")
    print("-----------------------------------------------------------------------")
    
    for rango in range(5):
        # TF-IDF
        if rango < len(tfidf_indices):
            tfidf_doc_idx = tfidf_indices[rango]
            tfidf_puntaje = tfidf_puntajes[rango]
            tfidf_text = corpus['text'].iloc[tfidf_doc_idx][:40]
            tfidf_str = f"{rango+1}. Doc {tfidf_doc_idx} ({tfidf_puntaje:.4f})"
        else:
            tfidf_str = ""
        
        # BM25
        if rango < len(bm25_indices):
            bm25_doc_idx = bm25_indices[rango]
            bm25_score = bm25_puntajes[rango]
            bm25_text = corpus['text'].iloc[bm25_doc_idx][:40]
            bm25_str = f"{rango+1}. Doc {bm25_doc_idx} ({bm25_score:.4f})"
        else:
            bm25_str = ""
        
        print(f"{tfidf_str:<50} | {bm25_str:<50}")

-------------------------
Comparacion de resultados
-------------------------

-----------------------------------------------
QUERY 1: 'machine learning artificial intelligence'

-----------------------------------------------

TF-IDF                                             | BM25                                              
-----------------------------------------------------------------------
1. Doc 10547 (0.7592)                              | 1. Doc 10547 (21.6419)                            
2. Doc 4733 (0.5932)                               | 2. Doc 8528 (19.5934)                             
3. Doc 9538 (0.5210)                               | 3. Doc 5991 (19.0860)                             
4. Doc 3905 (0.4879)                               | 4. Doc 9538 (18.8842)                             
5. Doc 7900 (0.4833)                               | 5. Doc 6569 (18.6432)                             

-----------------------------------------------
QUERY 2: 'python programmi

Analisis de ejemplo para la Query 1:
 
 - Los documentos recuperados por ambos modelos son 2 especificamen el Doc 10547 y el Doc 9538
 - Los documentos recuperados en TDF-IDF son los documentos 3905, 4733, 7900
 - Los documentos recuperados en BM25 son los documentos 5991, 6569, 8528

#### Conclusiones
1. TF-IDF se basa en la frecuencia de terminos y su rareza en el corpus
2. BM25 considera ademas la longitud del documento y saturacion de terminos
3. Las diferencias en los rankings reflejan enfoques distintos de relevancia
4. Documentos comunes indican que ambos modelos reconocen cierta relevancia
5. Documentos unicos muestran fortalezas especificas de cada algoritmo